This project trains a Convolutional Neural Network (CNN) to detect emotions from facial images using PyTorch.  


It covers data preprocessing, model training with early stopping, and evaluation on a test set.

How to Run

1. Install the required packages (`requirements.txt`).
2. Download and extract the dataset into the correct folders (`archive/train` and `archive/test`).
3. Run each cell in order to train and evaluate the model.

Project Structure

- Data loading and preprocessing
- Model definition
- Training loop with early stopping
- Evaluation on test data

Feel free to use or modify this notebook for your own experiments!



First we will import our modules we will need:

In [13]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2,transforms
from torchvision.datasets import ImageFolder
import os
import cv2
import torch.nn.functional as F
from torch.optim import Adam 
import random
import matplotlib.pyplot as plt


Next we will do our data preprocessing and preparation:

In [15]:
#loading the data
path = "archive"
train_dir = os.path.join(path, "train")
test_dir = os.path.join(path, "test")
test_data = ImageFolder(root=test_dir)


train_data_transforms = transforms.Compose([transforms.RandomResizedCrop(size=(48,48)), 
transforms.RandomHorizontalFlip(p=0.5), 
transforms.RandomRotation(0.2), transforms.ColorJitter(brightness=0.2, contrast=0.2, 
saturation=0.2, hue=0.1), 
transforms.RandomCrop(size=(48, 48), padding=4),
transforms.Grayscale(num_output_channels=1),
 transforms.ToTensor(),
transforms.Normalize(mean=[0.4870],std=[0.2284])]
)

train_data = ImageFolder(root=train_dir,transform=train_data_transforms)


train, val= random_split(train_data,[0.8,0.2]) #we want 80% training 20% for val
train_data_loader = DataLoader(train, batch_size=32, shuffle= True)
val_data_loader =  DataLoader(val, batch_size=32, shuffle= False)

test_data_transforms = transforms.Compose([
transforms.Grayscale(num_output_channels=1), transforms.ToTensor(),
transforms.Normalize(mean=[0.4870], std=[0.2284])])


# Now we have the DataLoader 
test_data = ImageFolder(root=test_dir,transform= test_data_transforms)
test_data_loader = DataLoader(test_data, batch_size = 32, shuffle=False)


Next we will create the Custom CNN.

In [16]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size= 3)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride = 2)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels= 128,kernel_size = 3 )
        self.pool2 =  nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.conv3 = nn.Conv2d(in_channels = 128, out_channels=256, kernel_size=3)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3)

        self.conv5 = nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=3)
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv6 = nn.Conv2d(in_channels=1024, out_channels=2048, kernel_size=3)


        self.fc1 = nn.Linear(in_features=51200,out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features = 64)
        self.fc3 = nn.Linear(in_features=64, out_features=7)

    def forward(self, x):
        x = F.relu(self.conv1(x))

        x = F.relu(self.conv2(x))

        x = self.pool2(x)

        x = F.relu(self.conv3(x))

        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        x =  F.relu(self.conv5(x))


        x = F.relu(self.conv6(x))

        x = torch.flatten(x,1) #flatten all dimensions except the first one 32 x (channels,length,width,)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.4)
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=0.4)

        x= self.fc3(x)

        return x
    

cnnModel = CNNModel()


We will also initialize the early stopping implementation.

In [17]:
patience = 50
delta = 0.01 #min change in loss we care about
best_loss = float('inf')
no_improv_counter = 0   

class EarlyStopping:
    def __init__(self,patience=10, delta=0, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss =None
        self.no_improv_counter = 0
        self.stop_training=False

    def check_early_stop(self, loss):
        if self.best_loss is None or loss < self.best_loss - self.delta: #loss is smaller
            self.best_loss = loss
            self.no_improv_counter = 0
            return True

        else:
            self.no_improv_counter +=1
            if self.no_improv_counter >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print(f"Early stopping triggered after {self.patience} epochs without improvement.")
            return False

early_stopping = EarlyStopping(patience=patience, delta=delta, verbose=True)

Next up is the training process, we will set a seed, so we can reproduce the same result across many runs.

In [ ]:
device =  'cuda' if torch.cuda.is_available() else 'cpu'
cnnModel.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnnModel.parameters(), lr=0.0001)
seed = 10
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)    
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
####train and validation
epochs = 300
total_loss,total_training, correct_training = 0, 0, 0
valid_loss, total_val_values, correct_val_values = 0,0,0
train_loss_list = []
val_loss_list = []

for epoch in range(epochs):
    cnnModel.train()
    total_training,correct_training, total_loss = 0,0,0

    for index, batch in enumerate(train_data_loader):
        
        image_inputs, labels = batch
        image_inputs, labels = image_inputs.to(device), labels.to(device)
        optimizer.zero_grad() #added to remove gradients from previous batches
        outputs =  cnnModel(image_inputs) #takes in the batched data and gives the classes
        loss = criterion(outputs, labels)
        loss.backward() #does backpropogation
        optimizer.step() #updates the weights
        total_loss += loss.item()  #converts the loss tensor into a number from each batch
        _, predicted = torch.max(outputs,1)
        total_training += labels.size(0)
        correct_training += (predicted==labels).sum().item()
    train_loss_list.append(total_loss/len(train_data_loader))

    valid_loss ,total_val_values, correct_val_values = 0,0, 0
    acc = 100 * (correct_training / total_training)

    avg_loss = total_loss/len(train_data_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {acc:.2f}% ")
    

    cnnModel.eval()
    for index, batch in enumerate(val_data_loader):


            val_image, val_label = batch
            val_image, val_label = val_image.to(device), val_label.to(device)
            with torch.no_grad(): #no gradients
                val_outputs = cnnModel(val_image)
                val_loss = criterion(val_outputs,val_label)
                valid_loss += val_loss.item()
                _, val_predicted = torch.max(val_outputs,1)
                correct_val_values +=(val_predicted == val_label).sum().item()
            total_val_values  += val_label.size(0)
        
    val_acc = 100 *(correct_val_values/total_val_values)

    val_loss_list.append(valid_loss/len(val_data_loader))
    validation_loss = valid_loss/len(val_data_loader)

    print(f" Epoch [{epoch+1}/{epochs}], Validation Loss: {validation_loss:.4f}, Validation Accuracy: {val_acc:.2f}% ")
    # Check early stopping and save checkpoint if loss improved
    
    if early_stopping.check_early_stop(validation_loss):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': cnnModel.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': early_stopping.best_loss,
        }
        torch.save(checkpoint, 'best_model2.pth')
        print("Model checkpoint saved.")
    
    # Check if we should stop training
    if early_stopping.stop_training:
        print(f"Training stopped early at epoch {epoch+1}")
        break

    total_loss = 0 #resets loss before next epoch

print("Training complete!")



        

Epoch [1/300], Loss: 1.8379, Accuracy: 23.00% 
 Epoch [1/300], Validation Loss: 1.8230, Validation Accuracy: 25.52% 
Model checkpoint saved.
Epoch [2/300], Loss: 1.8164, Accuracy: 23.94% 
 Epoch [2/300], Validation Loss: 1.7925, Validation Accuracy: 26.06% 
Model checkpoint saved.
Epoch [3/300], Loss: 1.7787, Accuracy: 25.58% 
 Epoch [3/300], Validation Loss: 1.7457, Validation Accuracy: 28.79% 
Model checkpoint saved.
Epoch [4/300], Loss: 1.7233, Accuracy: 29.24% 
 Epoch [4/300], Validation Loss: 1.6959, Validation Accuracy: 30.94% 
Model checkpoint saved.
Epoch [5/300], Loss: 1.6865, Accuracy: 31.06% 
 Epoch [5/300], Validation Loss: 1.6739, Validation Accuracy: 33.53% 
Model checkpoint saved.
Epoch [6/300], Loss: 1.6430, Accuracy: 33.02% 
 Epoch [6/300], Validation Loss: 1.6094, Validation Accuracy: 35.05% 
Model checkpoint saved.
Epoch [7/300], Loss: 1.6134, Accuracy: 35.33% 
 Epoch [7/300], Validation Loss: 1.5868, Validation Accuracy: 37.31% 
Model checkpoint saved.
Epoch [8/300]

Finally we wil evaluate the model on the test set.

In [ ]:
cnn_loaded = CNNModel()
cnn_loaded.load_state_dict(torch.load('best_model2.pth')['model_state_dict'])
cnn_loaded.to(device)
cnn_loaded.eval()
test_loss_list = []

correct = 0
total = 0 
with torch.no_grad(): #cant change weights
    for data in test_data_loader:
        images_inputs, labels = data
        images_inputs, labels = images_inputs.to(device), labels.to(device)
        outputs = cnn_loaded(images_inputs)
        # print(outputs)
        _, predicted = torch.max(outputs.data,1) #grabs the highest probability in each tensor(columns)
        # print(_)
        total += labels.size(0)  #how many images are in the batch
        correct += (predicted==labels).sum().item() #.item coverts to tensor 
        test_loss = criterion(outputs,labels)
        test_loss_list.append(test_loss.item())
        
print(f"Accuracy of the model on the test images: {100*correct/total} %")


Additional: Plots to see the comparison between the train and test losses.

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(train_loss_list,label='Training Loss')
plt.plot(test_loss_list, label="Test Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Test Loss")
plt.legend()
plt.show()

